# Chapter 8: Data Transformation: Strings, Categoricals, and Binning

Companion notebook for *Master Data Cleaning: A Hands-On Beginner's Guide with Python, pandas & NumPy*.

Run the cells from top to bottom (**Kernel → Restart Kernel and Run All Cells**). The data files this chapter uses are in the same folder as this notebook.

## 8.1 Renaming Columns

In [1]:
import pandas as pd
import numpy as np

employees = pd.read_csv('employee_data.csv')
employees.columns

Index(['Employee ID', ' First Name ', 'Department', 'Annual Salary'], dtype='str')

### `df.columns`

In [2]:
employees.columns = ['employee_id', 'first_name', 'department',
                     'annual_salary']
employees

,employee_id,first_name,department,annual_salary
0,101,Alice,HR,55000
1,102,Bob,IT,72000
2,103,Charlie,Sales,48000


In [3]:
employees = pd.read_csv('employee_data.csv')
employees.columns = (employees.columns.str.strip()
                     .str.lower()
                     .str.replace(' ', '_'))
employees.columns

Index(['employee_id', 'first_name', 'department', 'annual_salary'], dtype='str')

### `df.rename()`

In [4]:
employees.rename(columns={'employee_id': 'emp_id',
                          'annual_salary': 'salary'})

,emp_id,first_name,department,salary
0,101,Alice,HR,55000
1,102,Bob,IT,72000
2,103,Charlie,Sales,48000


In [5]:
employees = employees.rename(columns={'employee_id': 'emp_id',
                                      'annual_salary': 'salary'})
employees.columns

Index(['emp_id', 'first_name', 'department', 'salary'], dtype='str')

In [6]:
employees.rename(columns=str.upper)

,EMP_ID,FIRST_NAME,DEPARTMENT,SALARY
0,101,Alice,HR,55000
1,102,Bob,IT,72000
2,103,Charlie,Sales,48000


In [7]:
employees.rename(columns=lambda name: name.replace('_', ' ').title())

,Emp Id,First Name,Department,Salary
0,101,Alice,HR,55000
1,102,Bob,IT,72000
2,103,Charlie,Sales,48000


## 8.2 The `.str` Accessor

In [8]:
customers = pd.read_csv('customer_data.csv')
customers

,customer_id,city,email,status,full_name
0,1,New York,alice@email.com,Active,Alice Johnson
1,2,los angeles,BOB@EMAIL.COM,active,Bob Smith
2,3,Chicago,charlie@email.com,NaN,Charlie Williams
3,4,houston,DANA@email.com,Inactive,Dana Brown


In [9]:
customers.dtypes

customer_id    int64
city             str
email            str
status           str
full_name        str
dtype: object

### `s.str.lower()`, `s.str.upper()` and `s.str.title()`

In [10]:
cities = pd.DataFrame({
    'city': customers['city'],
    'lower': customers['city'].str.lower(),
    'title': customers['city'].str.title(),
})
cities

,city,lower,title
0,New York,new york,New York
1,los angeles,los angeles,Los Angeles
2,Chicago,chicago,Chicago
3,houston,houston,Houston


In [11]:
customers['email'].str.upper()

0      ALICE@EMAIL.COM
1        BOB@EMAIL.COM
2    CHARLIE@EMAIL.COM
3       DANA@EMAIL.COM
Name: email, dtype: str

### `s.str.contains()`, `s.str.startswith()` and `s.str.endswith()`

In [12]:
customers[customers['city'].str.endswith('o')]

,customer_id,city,email,status,full_name
2,3,Chicago,charlie@email.com,NaN,Charlie Williams


In [13]:
has_act = customers['status'].str.contains('act', case=False,
                                           na=False)
customers.loc[has_act, ['customer_id', 'status']]

,customer_id,status
0,1,Active
1,2,active
3,4,Inactive


### `s.str.split()`

In [14]:
customers[['first_name', 'last_name']] = (
    customers['full_name'].str.split(' ', n=1, expand=True)
)
customers[['full_name', 'first_name', 'last_name']]

,full_name,first_name,last_name
0,Alice Johnson,Alice,Johnson
1,Bob Smith,Bob,Smith
2,Charlie Williams,Charlie,Williams
3,Dana Brown,Dana,Brown


### Missing values

The next cell raises an error on purpose.

In [15]:
customers['status'].apply(lambda value: value.lower())

AttributeError: 'float' object has no attribute 'lower'

In [16]:
customers['status'].str.lower()

0      active
1      active
2         NaN
3    inactive
Name: status, dtype: str

## 8.3 Cleaning Text with `lower()`, `strip()` and `replace()`

In [17]:
products = pd.read_csv('products.csv')
products['category'].unique().tolist()

[' Electronics', 'Furniture ', 'electronics', 'Furniture', 'ELECTRONICS']

### `s.str.lower()`

In [18]:
products['category'].str.lower().unique().tolist()

[' electronics', 'furniture ', 'electronics', 'furniture']

### `s.str.strip()`

In [19]:
products['category'] = products['category'].str.strip().str.lower()
products['status'] = products['status'].str.strip().str.lower()
products['category'].unique().tolist()

['electronics', 'furniture']

In [20]:
products['price'] = products['price'].str.strip('$')
products

,product_id,category,status,price
0,P01,electronics,active,49.99
1,P02,furniture,active,129.00
2,P03,electronics,inactive,9.99
3,P04,furniture,active,75.50
4,P05,electronics,active,19.99


### `s.str.replace()`

In [21]:
contacts = pd.read_csv('customer_contacts.csv')
contacts

,customer_id,phone,paying_customer
0,1,(555) 123-4567,Y
1,2,555-987-6543,N
2,3,(555) 456-7890,Y
3,4,555/321/9876,Y


In [22]:
contacts['phone'].str.replace('-', '')

0    (555) 1234567
1       5559876543
2    (555) 4567890
3     555/321/9876
Name: phone, dtype: str

### Replacing with a regular expression

In [23]:
contacts['phone'] = contacts['phone'].str.replace(r'[()\s/-]', '',
                                                  regex=True)
contacts

,customer_id,phone,paying_customer
0,1,5551234567,Y
1,2,5559876543,N
2,3,5554567890,Y
3,4,5553219876,Y


### `s.replace()` for whole values

In [24]:
contacts['paying_customer'] = contacts['paying_customer'].replace(
    {'Y': 'Yes', 'N': 'No'}
)
contacts

,customer_id,phone,paying_customer
0,1,5551234567,Yes
1,2,5559876543,No
2,3,5554567890,Yes
3,4,5553219876,Yes


## 8.4 Extracting and Validating Text

### `s.str.extract()`

In [25]:
emails = pd.DataFrame({
    'customer_id': [1, 2, 3, 4],
    'email': ['alice@gmail.com', 'bob@yahoo.com',
              'charlie@hotmail', 'dana@company.com'],
})
emails[['username', 'domain']] = emails['email'].str.extract(
    r'([a-z]+)@([a-z]+\.com)'
)
emails

,customer_id,email,username,domain
0,1,alice@gmail.com,alice,gmail.com
1,2,bob@yahoo.com,bob,yahoo.com
2,3,charlie@hotmail,NaN,NaN
3,4,dana@company.com,dana,company.com


### Extracting numbers from text

In [26]:
stock = pd.DataFrame({
    'product': ['Widget', 'Gadget', 'Tool', 'Device'],
    'description': ['Widget - 25 units', 'Gadget - 100 units',
                    'Tool - 5 units', 'Device - 50 units'],
})
stock['quantity'] = stock['description'].str.extract(
    r'(\d+) units', expand=False
)
stock

,product,description,quantity
0,Widget,Widget - 25 units,25
1,Gadget,Gadget - 100 units,100
2,Tool,Tool - 5 units,5
3,Device,Device - 50 units,50


In [27]:
stock.dtypes

product        str
description    str
quantity       str
dtype: object

In [28]:
stock['quantity'] = pd.to_numeric(stock['quantity'])
stock['quantity'].sum()

np.int64(180)

### Validating a format with `s.str.contains()`

In [29]:
orders = pd.DataFrame({
    'order_id': [1, 2, 3, 4, 5],
    'order_code': ['ORD-2024-001', 'ord-2024-002', 'ORDER-2024-003',
                   'INV-2024-001', 'ORD-24-004'],
})
valid = orders['order_code'].str.contains(r'^ORD-\d{4}-\d{3}$',
                                          case=False)
orders[~valid]

,order_id,order_code
2,3,ORDER-2024-003
3,4,INV-2024-001
4,5,ORD-24-004


### A few patterns worth keeping

In [30]:
samples = pd.Series(['Order 25 shipped', 'Price 49.99',
                     'sam@shop.co.uk', '2024-03-15',
                     '(555) 123-4567'])
pd.DataFrame({
    'digits': samples.str.extract(r'(\d+)', expand=False),
    'decimal': samples.str.extract(r'(\d+\.\d+)', expand=False),
    'domain': samples.str.extract(r'@(.+)$', expand=False),
    'is_date': samples.str.contains(r'^\d{4}-\d{2}-\d{2}$'),
    'only_digits': samples.str.replace(r'\D', '', regex=True),
})

,digits,decimal,domain,is_date,only_digits
0,25,NaN,NaN,False,25
1,49,49.99,NaN,False,4999
2,NaN,NaN,shop.co.uk,False,
3,2024,NaN,NaN,True,20240315
4,555,NaN,NaN,False,5551234567


## 8.5 Converting Data Types

### Diagnosing types with `df.info()`

In [31]:
sales = pd.DataFrame({
    'product_id': ['001', '002', '003', '004'],
    'sales': ['1,250', '2,340', '875', '1,560'],
    'region': ['North', 'South', 'North', 'East'],
    'quantity': [10.0, 15.0, None, 20.0],
})
sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   product_id  4 non-null      str    
 1   sales       4 non-null      str    
 2   region      4 non-null      str    
 3   quantity    3 non-null      float64
dtypes: float64(1), str(3)
memory usage: 309.0 bytes


### `s.astype()`

The next cell raises an error on purpose.

In [32]:
sales['sales'].astype('float64')

ValueError: could not convert string to float: '1,250'

In [33]:
sales['sales'] = sales['sales'].str.replace(',', '').astype('float64')
sales['sales']

0    1250.0
1    2340.0
2     875.0
3    1560.0
Name: sales, dtype: float64

In [34]:
(sales['sales'] / 1000).astype('int64')

0    1
1    2
2    0
3    1
Name: sales, dtype: int64

In [35]:
sales['quantity'].astype('Int64')

0      10
1      15
2    <NA>
3      20
Name: quantity, dtype: Int64

### `pd.to_numeric()`

In [36]:
prices = pd.Series(['49.99', '129.00', 'unknown', '75.50'])
pd.to_numeric(prices, errors='coerce')

0     49.99
1    129.00
2       NaN
3     75.50
dtype: float64

### Saving memory with `category`

In [37]:
sales['region'] = sales['region'].astype('category')
sales['region']

0    North
1    South
2    North
3     East
Name: region, dtype: category
Categories (3, str): ['East', 'North', 'South']

In [38]:
regions = pd.Series(['North', 'South', 'East', 'West'] * 2500)
regions.memory_usage(deep=True)

125132

In [39]:
regions.astype('category').memory_usage(deep=True)

10183

### `pd.to_datetime()`

In [40]:
dates = pd.Series(['2024-01-15', 'not a date', '2024-03-10', None])
pd.to_datetime(dates, errors='coerce')

0   2024-01-15
1          NaT
2   2024-03-10
3          NaT
dtype: datetime64[us]

## 8.6 Binning with `pd.cut()` and `pd.qcut()`

### `pd.cut()`

In [41]:
ages = pd.Series([22, 25, 28, 32, 35, 38, 42, 45, 48, 52, 55, 58,
                  62, 65])
age_groups = pd.cut(ages, bins=[18, 30, 45, 60, 100],
                    labels=['Early career', 'Mid career',
                            'Senior', 'Retirement age'])
age_groups

0       Early career
1       Early career
2       Early career
3         Mid career
4         Mid career
5         Mid career
6         Mid career
7         Mid career
8             Senior
9             Senior
10            Senior
11            Senior
12    Retirement age
13    Retirement age
dtype: category
Categories (4, str): ['Early career' < 'Mid career' < 'Senior' < 'Retirement age']

In [42]:
pd.cut(ages, bins=[18, 30, 45, 60, 100]).value_counts(sort=False)

(18, 30]     3
(30, 45]     5
(45, 60]     4
(60, 100]    2
Name: count, dtype: int64

In [43]:
pd.cut(ages, bins=4).value_counts(sort=False)

(21.957, 32.75]    4
(32.75, 43.5]      3
(43.5, 54.25]      3
(54.25, 65.0]      4
Name: count, dtype: int64

In [44]:
pd.cut(pd.Series([18, 25, 67]), bins=[18, 30, 45, 60])

0             NaN
1    (18.0, 30.0]
2             NaN
dtype: category
Categories (3, interval[int64, right]): [(18, 30] < (30, 45] < (45, 60]]

### `pd.qcut()`

In [45]:
incomes = pd.Series([25000, 28000, 30000, 32000, 35000, 38000, 42000,
                     45000, 48000, 55000, 65000, 85000, 120000,
                     250000])
income_bands = pd.qcut(incomes, q=4,
                       labels=['Low', 'Lower middle',
                               'Upper middle', 'High'])
income_bands.value_counts(sort=False)

Low             4
Lower middle    3
Upper middle    3
High            4
Name: count, dtype: int64

In [46]:
pd.qcut(incomes, q=4).value_counts(sort=False)

(24999.999, 32750.0]    4
(32750.0, 43500.0]      3
(43500.0, 62500.0]      3
(62500.0, 250000.0]     4
Name: count, dtype: int64

### Choosing between `cut()` and `qcut()`

In [47]:
skewed = pd.Series([10, 12, 15, 18, 20, 25, 30, 40, 60, 100, 150,
                    200])
pd.DataFrame({
    'cut': pd.cut(skewed, bins=4).value_counts(sort=False).values,
    'qcut': pd.qcut(skewed, q=4).value_counts(sort=False).values,
}, index=['bin 1', 'bin 2', 'bin 3', 'bin 4'])

,cut,qcut
bin 1,8,3
bin 2,2,3
bin 3,1,3
bin 4,1,3


### Binning with `groupby()`

In [48]:
staff = pd.DataFrame({
    'employee_id': range(1, 11),
    'age': [24, 28, 32, 35, 38, 42, 45, 52, 58, 63],
    'sales': [45000, 52000, 61000, 58000, 67000, 72000, 68000,
              75000, 71000, 65000],
})
staff['age_group'] = pd.cut(staff['age'], bins=[20, 35, 50, 65],
                            labels=['Junior', 'Mid-level', 'Senior'])
staff.groupby('age_group', observed=False)['sales'].mean().round()

age_group
Junior       54000.0
Mid-level    69000.0
Senior       70333.0
Name: sales, dtype: float64

## 8.7 Putting It All Together

In [49]:
orders = pd.read_csv('orders_raw.csv')

orders.columns = (orders.columns.str.strip()
                  .str.lower()
                  .str.replace(' ', '_'))

orders['customer_name'] = (orders['customer_name'].str.strip()
                           .str.title())
orders['status'] = orders['status'].str.strip().str.lower()
orders['phone'] = orders['phone'].str.replace(r'\D', '', regex=True)

orders['sales'] = pd.to_numeric(
    orders['sales'].str.replace(',', ''), errors='coerce'
)
orders['region'] = orders['region'].astype('category')

orders['age_group'] = pd.cut(orders['age'], bins=[18, 30, 45, 60, 100],
                             labels=['18-30', '31-45', '46-60', '61+'])
orders

,order_id,customer_name,status,phone,sales,region,age,age_group
0,1001,Alice Tan,shipped,5551234567,1250.0,North,34,31-45
1,1002,Bob Lee,pending,5559876543,875.0,South,52,46-60
2,1003,Carol Ng,shipped,5554567890,2340.0,North,27,18-30
3,1004,Dev Raj,pending,5553219876,NaN,East,61,61+
4,1005,Ella Wong,cancelled,5552223333,1560.0,South,45,31-45


In [50]:
orders.dtypes

order_id            int64
customer_name         str
status                str
phone                 str
sales             float64
region           category
age                 int64
age_group        category
dtype: object